# 05-02 Inspect Runnables: 체인 내부 들여다보기

## 학습 목표

이 노트북을 완료하면 다음을 할 수 있어요:

- `get_graph()`로 체인의 노드(Node)와 엣지(Edge) 구조를 확인하는 방법을 설명할 수 있어요
- `print_ascii()`로 체인 실행 흐름을 텍스트 다이어그램으로 시각화할 수 있어요
- `get_prompts()`로 체인 안에 포함된 프롬프트 내용을 확인하고 디버깅에 활용할 수 있어요

## 사전 지식

이 노트북을 시작하기 전에 다음 개념을 알고 있으면 좋아요:

- LCEL 파이프라인 구성 방법 (`|` 연산자)
- `RunnablePassthrough`와 `RunnableParallel`의 역할
- RAG 체인의 기본 구조

---

LCEL로 `Runnable`을 생성한 후에는 종종 이를 검사하여 어떤 일이 일어나고 있는지 더 잘 파악하고 싶을 거예요.

이 노트북에서는 체인의 구조를 검사하는 세 가지 방법을 다뤄요:

- **그래프 구조 확인**: `get_graph()`로 체인의 노드와 엣지 확인
- **그래프 시각화**: `print_ascii()`로 ASCII 형식으로 출력
- **프롬프트 확인**: `get_prompts()`로 사용된 프롬프트 확인

In [ ]:
# API 키를 환경변수로 관리하기 위한 설정 파일
from dotenv import load_dotenv

# API 키 정보 로드
load_dotenv()


In [ ]:
# ============================================================
# TODO: get_graph()로 검사할 RAG 체인을 구성하세요
# 힌트: {"context": retriever, "question": RunnablePassthrough()} | prompt | model | StrOutputParser()
# 예상 결과: "✅ RAG 체인 생성 완료" 출력
# ============================================================

from langchain_community.vectorstores import FAISS
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

# 1단계: 예제 데이터로 FAISS 벡터 저장소 생성
vectorstore = FAISS.from_texts(
    ["파이썬은 데이터 과학과 머신러닝에 널리 사용되는 프로그래밍 언어입니다."],
    embedding=OpenAIEmbeddings(),
)

# 2단계: 벡터 저장소를 검색기로 변환
# as_retriever(): 벡터 저장소 → Runnable 검색기
retriever = vectorstore.as_retriever()

# 3단계: RAG 프롬프트 템플릿 정의
template = """다음 컨텍스트를 바탕으로 질문에 답변해주세요.
{context}  

질문: {question}"""

prompt = ChatPromptTemplate.from_template(template)

# ChatOpenAI 모델 초기화
model = ChatOpenAI(model="gpt-4o-mini")

# 4단계: RAG 체인 생성
# get_graph()로 내부 구조를 검사할 수 있도록 체인 저장


## 1. 그래프 구조 확인

`chain.get_graph()` 메서드는 체인의 실행 그래프를 반환해요.

- **노드(Node)**: 체인의 각 단계(Retriever, Prompt, LLM, Parser 등)를 나타내요
- **엣지(Edge)**: 단계 간의 데이터 흐름 방향을 나타내요

이를 통해 체인의 전체 구조를 코드 없이도 파악할 수 있어요.

> **주의:** 노드 ID는 실행할 때마다 새로 생성되는 해시값이라 매번 달라져요. ID 자체보다 노드의 역할(소스와 타깃 관계)에 집중해요.

In [ ]:
# ============================================================
# TODO: chain.get_graph()를 호출하여 노드 목록을 출력하세요
# 힌트: graph = chain.get_graph(); nodes = graph.nodes
# 예상 결과: 노드 ID 해시값 8개 출력
# ============================================================

# ---------------------------------------------------
# 체인의 그래프에서 노드 확인
# ---------------------------------------------------


In [ ]:
# 체인의 그래프에서 엣지 확인


## 2. 그래프 시각화

`print_ascii()` 메서드를 사용하면 그래프를 ASCII 형식으로 시각화할 수 있어요.

출력 결과를 보면 `Parallel<context,question>` 블록 안에서 `VectorStoreRetriever`와 `Passthrough`가 동시에 실행되고, 그 결과가 합쳐진 뒤 `ChatPromptTemplate → ChatOpenAI → StrOutputParser` 순서로 이어지는 구조를 한눈에 확인할 수 있어요.

> **실무 팁:** 복잡한 RAG 파이프라인을 처음 디버깅할 때 `print_ascii()`를 먼저 실행해보면 예상치 못한 단계나 잘못 연결된 경로를 빠르게 찾을 수 있어요.

In [ ]:
# ============================================================
# TODO: chain.get_graph().print_ascii()로 체인 구조를 시각화하세요
# 힌트: chain.get_graph().print_ascii()
# 예상 결과: Parallel<context,question> → ChatPromptTemplate → ChatOpenAI → StrOutputParser 흐름
# ============================================================

# ---------------------------------------------------
# 체인의 그래프를 ASCII 형식으로 시각화
# ---------------------------------------------------


## 3. 프롬프트 확인

체인에서 사용되는 프롬프트를 확인하는 것은 디버깅이나 최적화에 매우 유용해요.

`chain.get_prompts()` 메서드는 체인에서 사용되는 모든 프롬프트 객체의 리스트를 반환해요.

> **실무 팁:** 멀티 체인 구조에서 `get_prompts()`를 실행하면 중첩된 모든 서브체인의 프롬프트를 한꺼번에 확인할 수 있어요. 프롬프트 변수명(`input_variables`)이 체인 입력 딕셔너리의 키와 일치하는지 빠르게 검증하는 데 유용해요.

In [ ]:
# ============================================================
# TODO: chain.get_prompts()로 체인에서 사용하는 프롬프트를 출력하세요
# 힌트: prompts = chain.get_prompts(); for p in prompts: print(p)
# 예상 결과: ChatPromptTemplate 1개, input_variables=['context', 'question']
# ============================================================

# ---------------------------------------------------
# 체인에서 사용되는 프롬프트 확인
# ---------------------------------------------------


## 마무리 요약

### 체인 검사 메서드 비교

| 메서드 | 반환값 | 주요 용도 |
|--------|--------|-----------|
| `chain.get_graph().nodes` | 노드 딕셔너리 | 체인 구성 요소 목록 확인 |
| `chain.get_graph().edges` | 엣지 리스트 | 데이터 흐름 방향 확인 |
| `chain.get_graph().print_ascii()` | 없음 (출력만) | 전체 구조 시각화 |
| `chain.get_prompts()` | 프롬프트 객체 리스트 | 프롬프트 내용 및 변수 검증 |

### 핵심 요점

- `get_graph()`는 체인의 실행 흐름을 그래프 형태로 돌려줘요
- `print_ascii()`는 병렬 실행 구조(`Parallel`)와 순차 실행 흐름을 한눈에 보여줘요
- `get_prompts()`는 프롬프트 변수와 내용을 빠르게 검증할 때 사용해요
- 노드 ID는 매 실행마다 달라지므로 구조 파악에만 활용해요

### 다음 노트북 예고

다음 노트북(`03-RunnableLambda-and-ChainDecorator.ipynb`)에서는 사용자가 직접 정의한 파이썬 함수를 LCEL 파이프라인에 연결하는 방법을 배워요. `RunnableLambda`와 `@chain` 데코레이터를 비교하면서 살펴볼게요.